### Day - 4 : Pandas

### DataFrame, Series, Index

- **DataFrame**: 2D labeled table (rows × columns). Like a spreadsheet or SQL table.
- **Series**: 1D labeled array; each column of a DataFrame is a Series.
- **Index**: Labels for rows (or columns). Enables label-based selection and alignment.

Key idea: operations align on labels, not just positions.


In [2]:
#Example:
import pandas as pd

df = pd.DataFrame({
    "age": [25, 30, 35, 40],
    "salary": [50000, 60000, 75000, 90000],
    "dept": ["eng", "eng", "pm", "eng"]
}, index=["a", "b", "c", "d"])

df

,age,salary,dept
a,25,50000,eng
b,30,60000,eng
c,35,75000,pm
d,40,90000,eng


### Selecting data

- `df[...]`:
  - Single column: `df["age"]` → Series
  - Multiple columns: `df[["age","salary"]]` → DataFrame
  - Boolean mask: `df[df["age"] > 30]` → filtered DataFrame

- `df.loc[...]`: label-based (rows and/or columns by label)
  - `df.loc["b"]` → row with index "b"
  - `df.loc[:, ["age","dept"]]` → all rows, specific columns
  - `df.loc[df["age"] > 30, ["salary","dept"]]`

- `df.iloc[...]`: position-based (integer locations)
  - `df.iloc[0]` → first row
  - `df.iloc[1:3, [0,2]]` → rows 1–2, columns 0 and 2

- `df.query(...)`: SQL-like filtering
  - `df.query("age > 30 and dept == 'eng'")`


In [18]:
# Column selection
print('Age column:\n', df["age"])
print('Age and Salary columns:\n', df[["age", "salary"]])

# Boolean mask
print('Rows where age > 30:\n', df[df["age"] > 30])

# loc
print('Row with index "b":\n', df.loc["b"])
print('Age and Dept columns:\n', df.loc[:, ["age", "dept"]])
print('Rows where age > 30, Salary and Dept columns:\n', df.loc[df["age"] > 30, ["salary", "dept"]])

# iloc
print('First row:\n', df.iloc[0])
print('Rows 1 to 2, Columns 0 and 2:\n', df.iloc[1:3, [0, 2]])

# query
print('Rows where age > 30 and dept == "eng":\n', df.query("age > 30 and dept == 'eng'"))


Age column:
 a    25
b    30
c    35
d    40
Name: age, dtype: int64
Age and Salary columns:
    age  salary
a   25   50000
b   30   60000
c   35   75000
d   40   90000
Rows where age > 30:
    age  salary dept  salary_by_dept   level
c   35   75000   pm    75000.000000     mid
d   40   90000  eng    66666.666667  senior
Row with index "b":
 age                         30
salary                   60000
dept                       eng
salary_by_dept    66666.666667
level                   senior
Name: b, dtype: object
Age and Dept columns:
    age dept
a   25  eng
b   30  eng
c   35   pm
d   40  eng
Rows where age > 30, Salary and Dept columns:
    salary dept
c   75000   pm
d   90000  eng
First row:
 age                         25
salary                   50000
dept                       eng
salary_by_dept    66666.666667
level                   junior
Name: a, dtype: object
Rows 1 to 2, Columns 0 and 2:
    age dept
b   30  eng
c   35   pm
Rows where age > 30 and dept == "eng":
    age

### Missing data

- Missing values are represented as `NaN` (for floats) or `pd.NA` (newer nullable types).
- Common methods:
  - `df.isna()` / `df.notna()` → boolean mask
  - `df.isna().sum()` → count missing per column
  - `df.dropna()` → drop rows/columns with missing values
  - `df.fillna(value)` → fill missing with a constant, mean, median, etc.


In [19]:
df2 = pd.DataFrame({
    "x": [1.0, np.nan, 3.0, 4.0],
    "y": [10, 20, np.nan, 40]
})

print('Missing values:\n', df2.isna())
print('Count of missing values:\n', df2.isna().sum())

print('Rows without missing values:\n', df2.dropna())
print('Filled missing values:\n', df2.fillna({"x": df2["x"].median(), "y": 0}))


Missing values:
        x      y
0  False  False
1   True  False
2  False   True
3  False  False
Count of missing values:
 x    1
y    1
dtype: int64
Rows without missing values:
      x     y
0  1.0  10.0
3  4.0  40.0
Filled missing values:
      x     y
0  1.0  10.0
1  3.0  20.0
2  3.0   0.0
3  4.0  40.0


### groupby, agg, transform

- **`groupby`**: split data into groups based on one or more keys.
- **`agg`**: aggregate each group to a single row (e.g. mean, sum).
- **`transform`**: compute group-wise values but broadcast back to original shape.

Use cases:
- Summaries by category (e.g. average salary by department).
- Creating features like “group mean” for each row.


In [20]:
df

# Group by department
g = df.groupby("dept")

# Agg: one row per dept
print('Group means:\n', g["salary"].mean())
print('Group aggregates:\n', g.agg({"salary": ["mean", "std"], "age": "median"}))

# Transform: add group mean as a new column
df["salary_by_dept"] = g["salary"].transform("mean")
print('DataFrame with group means:\n', df)


Group means:
 dept
eng    66666.666667
pm     75000.000000
Name: salary, dtype: float64
Group aggregates:
             salary                  age
              mean           std median
dept                                   
eng   66666.666667  20816.659995   30.0
pm    75000.000000           NaN   35.0
DataFrame with group means:
    age  salary dept  salary_by_dept   level
a   25   50000  eng    66666.666667  junior
b   30   60000  eng    66666.666667  senior
c   35   75000   pm    75000.000000     mid
d   40   90000  eng    66666.666667  senior


### pivot_table and crosstab

- **`pivot_table`**: reshape data; summarize values in a 2D grid.
  - Like Excel pivot tables.
  - `index` = rows, `columns` = columns, `values` = what to aggregate.

- **`crosstab`**: compute frequency tables (counts) between two or more categorical variables.


In [16]:
# Pivot: average salary by dept and (fake) level
df["level"] = ["junior", "senior", "mid", "senior"]

pt = df.pivot_table(
    values="salary",
    index="dept",
    columns="level",
    aggfunc="mean"
)
print('Pivot Table:\n', pt)

# Crosstab: counts of dept × level
ct = pd.crosstab(df["dept"], df["level"])
print('crosstab:\n', ct)



Pivot Table:
 level   junior      mid   senior
dept                            
eng    50000.0      NaN  75000.0
pm         NaN  75000.0      NaN
crosstab:
 level  junior  mid  senior
dept                      
eng         1    0       2
pm          0    1       0


### merge and concat

- **`merge`**: join two DataFrames on keys (like SQL JOIN).
  - `on`, `left_on`, `right_on`, `how` (`"inner"`, `"left"`, `"right"`, `"outer"`).

- **`concat`**: stack DataFrames vertically or horizontally.
  - `axis=0` → stack rows
  - `axis=1` → stack columns


In [15]:
df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "carol"]
})

df2 = pd.DataFrame({
    "id": [2, 3, 4],
    "score": [80, 90, 70]
})

# Merge (inner join on "id")
print('Inner Join:')
print(df1.merge(df2, on="id", how="inner"))
print('Left Join:')
print(df1.merge(df2, on="id", how="left"))
print('Right Join:')
print(df1.merge(df2, on="id", how="right"))

# Concat
print('Concatenate Rows:')
print(pd.concat([df1, df2], axis=0))        # rows; columns aligned by name
print('Concatenate Columns:')
print(pd.concat([df1, df2.set_index("id")], axis=1))  # columns


Inner Join:
   id   name  score
0   2    bob     80
1   3  carol     90
Left Join:
   id   name  score
0   1  alice    NaN
1   2    bob   80.0
2   3  carol   90.0
Right Join:
   id   name  score
0   2    bob     80
1   3  carol     90
2   4    NaN     70
Concatenate Rows:
   id   name  score
0   1  alice    NaN
1   2    bob    NaN
2   3  carol    NaN
0   2    NaN   80.0
1   3    NaN   90.0
2   4    NaN   70.0
Concatenate Columns:
    id   name  score
0  1.0  alice    NaN
1  2.0    bob    NaN
2  3.0  carol   80.0
3  NaN    NaN   90.0
4  NaN    NaN   70.0


### What is feature engineering?

Feature engineering = transforming raw data into features (columns) that models can use effectively.

Goals:
- Make patterns easier to learn.
- Encode domain knowledge.
- Handle non-linearities, interactions, and structure in the data.

Good features often matter more than the choice of model.

1. Derived features: Create new columns by combining or transforming existing ones.
2.	Binning (discretization): Turn a continuous variable into categories (bins).
3.	One-hot / dummy encoding: Convert categorical variables into binary columns.
4.	Aggregation-based features: Use group-wise statistics as features for each row.
5.	Interaction features: Explicitly model interactions between features.
6.	Scaling / normalization: Rescale numeric features to a common range or distribution.
Common methods:
- **Standardization**: subtract mean, divide by std → mean 0, std 1.
- **Min–max scaling**: scale to [0, 1].

Important for:
- Models using distances (k-NN, k-means).
- Regularized linear models (ridge, lasso).
- Gradient-based optimization (neural nets).


In [21]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "income": [0, 5000, 30000, 80000, 50000],
    "spend": [0, 4000, 20000, 60000, 45000],
    "age": [10, 17, 25, 40, 65]
})

# Derived features
df["log_income"] = np.log1p(df["income"])           # log(1 + income)
df["spend_ratio"] = df["spend"] / (df["income"] + 1)
df["is_adult"] = (df["age"] >= 18).astype(int)

df


,income,spend,age,log_income,spend_ratio,is_adult
0,0,0,10,0.000000,0.000000,0
1,5000,4000,17,8.517393,0.799840,0
2,30000,20000,25,10.308986,0.666644,1
3,80000,60000,40,11.289794,0.749991,1
4,50000,45000,65,10.819798,0.899982,1


In [ ]:
# Binning
df["age_bin"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["child","teen","young","mid","senior"]
)

df["income_bin"] = pd.qcut(
    df["income"],
    q=3,                  # 3 quantile-based bins
    labels=["low","med","high"]
)

df


,income,spend,age,log_income,spend_ratio,is_adult,age_bin,income_bin
0,0,0,10,0.000000,0.000000,0,child,low
1,5000,4000,17,8.517393,0.799840,0,teen,low
2,30000,20000,25,10.308986,0.666644,1,young,med
3,80000,60000,40,11.289794,0.749991,1,mid,high
4,50000,45000,65,10.819798,0.899982,1,senior,high


In [ ]:
# One-hot encoding
df2 = pd.DataFrame({
    "city": ["A", "B", "A", "C"],
    "gender": ["M", "F", "F", "M"],
    "age": [25, 30, 35, 40]
})

encoded = pd.get_dummies(df2, columns=["city", "gender"], drop_first=True)
encoded

,age,city_B,city_C,gender_M
0,25,False,False,True
1,30,True,False,False
2,35,False,False,False
3,40,False,True,True


In [24]:
#Aggregation with groupby and transform
df3 = pd.DataFrame({
    "user": [1, 1, 2, 2, 3],
    "amount": [100, 200, 50, 70, 300],
    "city": ["A", "A", "A", "A", "B"]
})

# Mean amount per user
df3["user_mean_amount"] = df3.groupby("user")["amount"].transform("mean")

# Mean amount per city
df3["city_mean_amount"] = df3.groupby("city")["amount"].transform("mean")

df3


,user,amount,city,user_mean_amount,city_mean_amount
0,1,100,A,150.0,105.0
1,1,200,A,150.0,105.0
2,2,50,A,60.0,105.0
3,2,70,A,60.0,105.0
4,3,300,B,300.0,300.0


In [25]:
#Scaling / Normalization
df5 = pd.DataFrame({
    "age": [10, 20, 30, 40, 50],
    "income": [20000, 40000, 60000, 80000, 100000]
})

# Standardization
df5_std = (df5 - df5.mean()) / df5.std()

# Min–max scaling
df5_minmax = (df5 - df5.min()) / (df5.max() - df5.min())

df5_std, df5_minmax


(        age    income
 0 -1.264911 -1.264911
 1 -0.632456 -0.632456
 2  0.000000  0.000000
 3  0.632456  0.632456
 4  1.264911  1.264911,
     age  income
 0  0.00    0.00
 1  0.25    0.25
 2  0.50    0.50
 3  0.75    0.75
 4  1.00    1.00)

## Titanic Pandas + Feature Engineering Exercises

Use the same `df` (Titanic dataset) for all exercises.


In [29]:
import pandas as pd
import numpy as np
from pathlib import Path

# Determine paths (works whether notebook is in repo root or nested)
notebook_path = Path.cwd()
repo_root = notebook_path
# If your notebook is inside `notebooks/`, go up one level:
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

data_dir = repo_root / "data"
titanic_path = data_dir / "titanic.csv"

print("Loading Titanic from:", titanic_path)

# Load CSV
df = pd.read_csv(titanic_path)

# Inspect columns to see exact names
print("Columns:", df.columns.tolist())
df.head()


Loading Titanic from: /Users/tarun/tarun/genai/code/gen-ai-test/week01/data/titanic.csv
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [30]:
# Rename to lowercase and standard names
df = df.rename(columns={
    "Survived": "survived",
    "Pclass": "pclass",
    "Age": "age",
    "SibSp": "sibsp",
    "Parch": "parch",
    "Fare": "fare",
    "Sex": "sex",
    "Embarked": "embarked"
})

# Create 'class' column similar to seaborn's version
df["class"] = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})

# Ensure numeric columns are numeric
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.columns.tolist())
df.head()


['PassengerId', 'survived', 'pclass', 'Name', 'sex', 'age', 'sibsp', 'parch', 'Ticket', 'fare', 'Cabin', 'embarked', 'class']


,PassengerId,survived,pclass,Name,sex,age,sibsp,parch,Ticket,fare,Cabin,embarked,class
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Third
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,First
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Third
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,First
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Third


### Exercise 1: Basic inspection & cleaning

**Goals:**

- Inspect shape, dtypes, and missing values.
- Drop unnecessary columns (e.g. `deck`, `embark_town`, `alive`, `who`, `adult_male`, `name`, `ticket`, `boat` if present).
- Fill missing `age` with median age.
- Fill missing `embarked` with the most common port.
- Handle missing `fare` (drop rows or fill with median).

In [31]:
# 1. Inspection
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:\n", df.dtypes)
print("Missing values:\n", df.isna().sum())

# 2. Drop unnecessary columns (adjust if your CSV has different extras)
cols_to_drop = [c for c in ["deck", "embark_town", "alive", "who", "adult_male", "name", "ticket", "boat"] 
                if c in df.columns]
df = df.drop(columns=cols_to_drop)

# 3. Fill missing age with median
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

# 4. Fill missing embarked with mode
embarked_mode = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(embarked_mode)

# 5. Handle missing fare (drop or fill)
df = df.dropna(subset=["fare"])  # or: df["fare"] = df["fare"].fillna(df["fare"].median())

print("Missing values after cleaning:\n", df.isna().sum())


Shape: (891, 13)
Columns: ['PassengerId', 'survived', 'pclass', 'Name', 'sex', 'age', 'sibsp', 'parch', 'Ticket', 'fare', 'Cabin', 'embarked', 'class']
Dtypes:
 PassengerId      int64
survived         int64
pclass           int64
Name               str
sex                str
age            float64
sibsp            int64
parch            int64
Ticket             str
fare           float64
Cabin              str
embarked           str
class              str
dtype: object
Missing values:
 PassengerId      0
survived         0
pclass           0
Name             0
sex              0
age            177
sibsp            0
parch            0
Ticket           0
fare             0
Cabin          687
embarked         2
class            0
dtype: int64
Missing values after cleaning:
 PassengerId      0
survived         0
pclass           0
Name             0
sex              0
age              0
sibsp            0
parch            0
Ticket           0
fare             0
Cabin          687
embarked

### Exercise 2: Selection, filtering, and `query`

**Goals:**

- Define feature columns (exclude `survived` and non-feature columns).
- Create:
  - `X_all = df[feature_cols]`
  - `y_all = df["survived"]`
- Filter to:
  - Female passengers aged 18–40.
- Using `query`, select:
  - Passengers in `class == "First"` with `fare > 50`.  
    (If `class` causes issues in `query`, use backticks: `` `class` `` or rename the column.)
- Use `loc` to get `["age", "fare", "survived"]` for passengers with `pclass == 3`.

In [33]:
# 1–2. Feature columns and X_all, y_all
feature_cols = ["pclass", "age", "sibsp", "parch", "fare", "sex", "embarked"]
X_all = df[feature_cols]
y_all = df["survived"]

# 3. Female passengers aged 18–40
df_female_young = df[
    (df["sex"] == "female") &
    (df["age"] >= 18) &
    (df["age"] <= 40)
]

# 4. Query: First class, fare > 50
df_first_rich = df.query('`class` == "First" and fare > 50')

# 5. loc: pclass == 3, columns age, fare, survived
df_third_class = df.loc[df["pclass"] == 3, ["age", "fare", "survived"]]

# Quick checks
print("Female 18–40 shape:", df_female_young.shape)
print("First class, fare>50 shape:", df_first_rich.shape)
print("Third class sample:\n", df_third_class.head())


Female 18–40 shape: (211, 13)
First class, fare>50 shape: (139, 13)
Third class sample:
     age     fare  survived
0  22.0   7.2500         0
2  26.0   7.9250         1
4  35.0   8.0500         0
5  28.0   8.4583         0
7   2.0  21.0750         0


### Exercise 3: `groupby`, `agg`, `transform`

**Goals:**

- Group by `class` and compute:
  - mean and std of `fare`
  - median of `age`
  - mean of `survived` (survival rate by class)
- Add a column `class_survival_rate` = mean survival for that passenger’s class (use `transform`).
- Add a column `age_zscore_by_class` = standardized age within each class.


In [36]:
# 1. Agg by class
class_stats = df.groupby("class").agg(
    fare_mean=("fare", "mean"),
    fare_std=("fare", "std"),
    age_median=("age", "median"),
    survival_rate=("survived", "mean")
)
print('Aggregated stats by class: \n',class_stats)

# 2. class_survival_rate column
df["class_survival_rate"] = df.groupby("class")["survived"].transform("mean")

# 3. age z-score by class
g = df.groupby("class")["age"]
df["age_zscore_by_class"] = (df["age"] - g.transform("mean")) / g.transform("std")

df[["class", "age", "age_zscore_by_class", "class_survival_rate", "survived"]].head()


Aggregated stats by class: 
         fare_mean   fare_std  age_median  survival_rate
class                                                  
First   84.154687  78.380373        35.0       0.629630
Second  20.662183  13.417399        28.0       0.472826
Third   13.675550  11.778142        28.0       0.242363


,class,age,age_zscore_by_class,class_survival_rate,survived
0,Third,22.0,-0.367615,0.242363,0
1,First,38.0,0.083758,0.629630,1
2,Third,26.0,0.006298,0.242363,1
3,First,35.0,-0.127776,0.629630,1
4,Third,35.0,0.847602,0.242363,0


### Exercise 4: Binning and derived features

**Goals:**

- Create `age_bin`:
  - bins: `[0, 12, 18, 35, 60, 100]`
  - labels: `["child","teen","young","mid","senior"]`
- Create `fare_bin` using quantiles (e.g. 4 bins: very_low, low, med, high).
- Create derived features:
  - `family_size = sibsp + parch + 1`
  - `is_alone = (family_size == 1).astype(int)`
  - `log_fare = log(1 + fare)`
  - `is_child = (age < 12).astype(int)`


In [37]:
# 1. Age bins
df["age_bin"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["child","teen","young","mid","senior"]
)
print('Age bins:\n', df[["age", "age_bin"]].head())
# 2. Fare bins (quantile-based)
df["fare_bin"] = pd.qcut(
    df["fare"],
    q=4,
    labels=["very_low","low","med","high"]
)
print('Fare bins:\n', df[["fare", "fare_bin"]].head())
# 3. Derived features
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)
df["log_fare"] = np.log1p(df["fare"])
df["is_child"] = (df["age"] < 12).astype(int)
print('Derived features:\n')
df[["age_bin","fare_bin","family_size","is_alone","log_fare","is_child"]].head()


Age bins:
     age age_bin
0  22.0   young
1  38.0     mid
2  26.0   young
3  35.0   young
4  35.0   young
Fare bins:
       fare  fare_bin
0   7.2500  very_low
1  71.2833      high
2   7.9250       low
3  53.1000      high
4   8.0500       low
Derived features:



,age_bin,fare_bin,family_size,is_alone,log_fare,is_child
0,young,very_low,2,0,2.110213,0
1,mid,high,2,0,4.280593,0
2,young,low,1,1,2.188856,0
3,young,high,2,0,3.990834,0
4,young,low,1,1,2.202765,0


### Exercise 5: One-hot encoding

**Goals:**

- One-hot encode:
  - `sex`
  - `embarked`
  - `age_bin`
  - `fare_bin`
- Use `drop_first=True`.
- Create a final feature matrix `X` that includes:
  - Numeric:  
    `pclass`, `age`, `sibsp`, `parch`, `log_fare`, `family_size`, `is_alone`, `is_child`, `age_zscore_by_class`, `class_survival_rate`
  - One-hot dummies for the categorical columns above.
- Keep `y = df["survived"]`.

In [38]:
# One-hot encode categoricals
X_cat = pd.get_dummies(
    df[["sex", "embarked", "age_bin", "fare_bin"]],
    drop_first=True
)

# Numeric features
num_cols = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "log_fare",
    "family_size",
    "is_alone",
    "is_child",
    "age_zscore_by_class",
    "class_survival_rate"
]

X_num = df[num_cols]

# Final feature matrix
X = pd.concat([X_num, X_cat], axis=1)
y = df["survived"]

print("Feature matrix shape:", X.shape)
X.head()


Feature matrix shape: (891, 20)


,pclass,age,sibsp,parch,log_fare,family_size,is_alone,is_child,age_zscore_by_class,class_survival_rate,sex_male,embarked_Q,embarked_S,age_bin_teen,age_bin_young,age_bin_mid,age_bin_senior,fare_bin_low,fare_bin_med,fare_bin_high
0,3,22.0,1,0,2.110213,2,0,0,-0.367615,0.242363,True,False,True,False,True,False,False,False,False,False
1,1,38.0,1,0,4.280593,2,0,0,0.083758,0.629630,False,False,False,False,False,True,False,False,False,True
2,3,26.0,0,0,2.188856,1,1,0,0.006298,0.242363,False,False,True,False,True,False,False,True,False,False
3,1,35.0,1,0,3.990834,2,0,0,-0.127776,0.629630,False,False,True,False,True,False,False,False,False,True
4,3,35.0,0,0,2.202765,1,1,0,0.847602,0.242363,True,False,True,False,True,False,False,True,False,False


### Exercise 6: Train/test split and leakage-safe engineering

**Goals:**

- Shuffle `df` and split into train (80%) and test (20%).
- On **train only**:
  - Compute median `age` and `fare` (for imputation).
  - Compute mean and std of each numeric feature for standardization.
- Apply:
  - Missing-value imputation using train medians to both train and test.
  - Standardization (fit on train, apply to train and test).
- One-hot encode categoricals consistently, aligning columns between train and test.
- End with:
  - `X_train`, `X_test` (fully numeric feature matrices)
  - `y_train`, `y_test`


In [39]:
# 1. Shuffle and split
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
split = int(0.8 * len(df_shuffled))

train = df_shuffled.iloc[:split].copy()
test = df_shuffled.iloc[split:].copy()

# 2. Compute stats on train only
age_median_train = train["age"].median()
fare_median_train = train["fare"].median()

num_cols_for_scale = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "log_fare",
    "family_size",
    "is_alone",
    "is_child",
    "age_zscore_by_class",
    "class_survival_rate"
]

train_mean = train[num_cols_for_scale].mean()
train_std = train[num_cols_for_scale].std()

# 3. Impute missing (if any) using train medians
train["age"] = train["age"].fillna(age_median_train)
test["age"] = test["age"].fillna(age_median_train)

train["fare"] = train["fare"].fillna(fare_median_train)
test["fare"] = test["fare"].fillna(fare_median_train)

# Recompute log_fare after imputation
train["log_fare"] = np.log1p(train["fare"])
test["log_fare"] = np.log1p(test["fare"])

# Standardize
X_train_num = (train[num_cols_for_scale] - train_mean) / train_std
X_test_num = (test[num_cols_for_scale] - train_mean) / train_std

# 4. One-hot encode categoricals consistently
X_cat_train = pd.get_dummies(
    train[["sex", "embarked", "age_bin", "fare_bin"]],
    drop_first=True
)
X_cat_test = pd.get_dummies(
    test[["sex", "embarked", "age_bin", "fare_bin"]],
    drop_first=True
)

# Align columns (in case some categories missing in test)
X_cat_train, X_cat_test = X_cat_train.align(X_cat_test, fill_value=0, axis=1)

X_train = pd.concat([X_train_num, X_cat_train], axis=1)
X_test = pd.concat([X_test_num, X_cat_test], axis=1)

y_train = train["survived"]
y_test = test["survived"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (712, 20)
X_test shape: (179, 20)


### Exercise 7: Logistic regression sanity-check

**Goals:**

- Train a logistic regression classifier on `X_train`, `y_train`.
- Evaluate:
  - Train accuracy
  - Test accuracy
  - Classification report on the test set
- Inspect model coefficients:
  - Create a DataFrame of `feature` vs `coef`.
  - Sort by absolute coefficient magnitude.
  - Identify the top 10 most influential features for predicting survival

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Define and fit the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 2. Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# 3. Accuracies
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

# 4. Classification report on test set
print("\nTest set classification report:\n")
print(classification_report(y_test, y_test_pred, digits=3))

# 5. Inspect coefficients (optional but useful)
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model.coef_[0]
}).sort_values("coef", key=abs, ascending=False)

print("\nTop 10 features by absolute coefficient:\n")
print(coef_df.head(10))


Train accuracy: 0.8103932584269663
Test accuracy: 0.8044692737430168

Test set classification report:

              precision    recall  f1-score   support

           0      0.856     0.833     0.844       114
           1      0.721     0.754     0.737        65

    accuracy                          0.804       179
   macro avg      0.788     0.794     0.791       179
weighted avg      0.807     0.804     0.805       179


Top 10 features by absolute coefficient:

         feature      coef
10      sex_male -2.465599
0         pclass -0.564138
2          sibsp -0.557027
13  age_bin_teen -0.436411
6       is_alone -0.429775
5    family_size -0.406086
4       log_fare  0.306772
17  fare_bin_low  0.283201
7       is_child  0.280214
1            age -0.240418
